<!-- context: VAAET/notebooks/phase_2_intelligence/02_traffic_state_classifier.ipynb
Etapa 2 del pipeline VAAET — Clasificación de estado de tráfico.
Consume telemetría de Etapa 1 (traffic_data) y produce clasificaciones.
ADR-008 documenta la decisión de TF/Keras. No modifica Etapa 1. -->

# Etapa 2 — Clasificador de Estado de Tráfico

La Etapa 1 de VAAET (notebook `01_legacy_collection.ipynb`) detecta vehículos, los rastrea y estima su velocidad. Cada minuto persiste un registro con velocidad promedio y conteos por tipo en la tabla `traffic_data` de PostgreSQL. Pero esos 9 campos crudos no responden la pregunta operacional que el sistema SISE necesita: **¿cuál es el estado actual del tráfico en el puente?**

Este notebook implementa la **capa de inteligencia** que transforma telemetría cruda en una clasificación de 4 estados operacionales:

| Estado | Código | Criterio de ingeniería |
|---|---|---|
| **Normal** | 0 | Flujo libre (default: todo lo que no cumpla criterios más severos) |
| **Reducido** | 1 | Flujo degradado: 5-40 km/h, 15-25 veh/min |
| **Atascado** | 2 | Congestión: <5 km/h, >25 veh/min, persistencia ≥2 min |
| **Accidente** | 3 | Evento disruptivo: ~0 km/h tras frenada brusca (delta < -20 km/h), persistencia ≥3 min |

**Arquitectura**: MLP tabular con TensorFlow/Keras (Fase 1), diseñado para evolucionar a LSTM con memoria temporal en Fase 2. Ver [ADR-008](../../docs/adr/ADR-008-tensorflow-keras-traffic-classifier.md) para el razonamiento completo.

In [ ]:
# Cell 0 — Setup de Entorno (Colab / Local)
# 
# En Google Colab el CWD es /content, no la raíz del repo.
# Esta celda clona o monta el repo y hace %cd al directorio
# del notebook para que las rutas relativas (../../models, etc.)
# resuelvan correctamente. En VS Code / local es un no-op.

import os

try:
    import google.colab  # type: ignore[import-untyped]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/titesen/vaaet.git"  
    REPO_DIR = "/content/vaaet"
    NB_DIR = os.path.join(REPO_DIR, "notebooks", "phase_2_intelligence")

    if not os.path.isdir(REPO_DIR):
        print("📦 Clonando repositorio VAAET...")
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
    else:
        print("📂 Repositorio ya presente, actualizando...")
        os.system(f"git -C {REPO_DIR} pull --ff-only")

    os.chdir(NB_DIR)
    print(f"✅ Colab CWD → {os.getcwd()}")
else:
    print(f"✅ Entorno local detectado — CWD: {os.getcwd()}")

In [ ]:
# Cell 1 — Dependencias e Imports
# 
# En Google Colab, TensorFlow ya está preinstalado. Solo se
# instalan las dependencias adicionales del pipeline.

import subprocess
import sys

def install_if_missing(package: str, import_name: str | None = None) -> None:
    """Instala un paquete si no está disponible en el entorno."""
    name = import_name or package
    try:
        __import__(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

# Dependencias que pueden no estar preinstaladas
install_if_missing("imbalanced-learn", "imblearn")
install_if_missing("sqlalchemy")
install_if_missing("psycopg2-binary", "psycopg2")
install_if_missing("seaborn")
install_if_missing("joblib")

# Core 
import numpy as np
import pandas as pd
import os
import getpass
from datetime import datetime

# TensorFlow / Keras 
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Scikit-learn 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# SMOTE 
from imblearn.over_sampling import SMOTE

# SQLAlchemy 
from sqlalchemy import create_engine, text

# Serialización 
import joblib

# Visualización 
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibilidad
RANDOM_SEED: int = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Rutas de artefactos 
MODEL_DIR: str = os.path.join("..", "..", "models", "intelligence")
DATA_DIR: str = os.path.join("..", "..", "data", "processed")
RAW_DIR: str = os.path.join("..", "..", "data", "raw")
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)

print(f"✅ Dependencias cargadas")
print(f"   TensorFlow {tf.__version__} | Pandas {pd.__version__} | NumPy {np.__version__}")
print(f"   Seed global: {RANDOM_SEED}")
print(f"   Artefactos → {os.path.abspath(MODEL_DIR)}")

## Fuente de Datos — Telemetría de Etapa 1

Los datos provienen de la tabla `traffic_data` en PostgreSQL (AWS RDS), producidos por la Etapa 1 de percepción. Cada registro representa un minuto de video procesado con 9 campos: velocidad promedio, conteos por tipo de vehículo (car, truck, bus, motorcycle, bicycle) y total.

El dataset real contiene ~2000 registros del backup `traffic_data.backup` (formato `pg_dump`). Para usar este notebook necesitas:

1. **Opción A** — Conexión directa a la instancia RDS donde se ejecutó Etapa 1
2. **Opción B** — Restaurar el backup en una instancia PostgreSQL local: `pg_restore -d vaaet data/raw/traffic_data.backup`

Las credenciales se obtienen por variables de entorno (`DB_HOST`, `DB_PORT`, `DB_NAME`, `DB_USER`, `DB_PASSWORD`) o input interactivo. Nunca se hardcodean ni se imprimen en outputs.

Al cargar con éxito desde la BD, se guarda una copia cruda en `data/raw/traffic_data_raw.csv` para permitir ejecuciones futuras sin conexión.

In [ ]:
# Cell 2 — Conexión a BD + Extracción de Telemetría

RAW_CSV_PATH: str = os.path.join("..", "..", "data", "raw", "traffic_data_raw.csv")


def get_db_config() -> dict[str, str]:
    """Obtiene configuración de BD desde env vars o input interactivo."""
    config = {
        "host": os.environ.get("DB_HOST", ""),
        "port": os.environ.get("DB_PORT", "5432"),
        "dbname": os.environ.get("DB_NAME", ""),
        "user": os.environ.get("DB_USER", ""),
        "password": os.environ.get("DB_PASSWORD", ""),
    }
    if not config["host"]:
        print("📋 Configuración de PostgreSQL (variables de entorno no encontradas)")
        config["host"] = input("   Host: ").strip()
        config["port"] = input("   Port [5432]: ").strip() or "5432"
        config["dbname"] = input("   Database: ").strip()
        config["user"] = input("   User: ").strip()
        config["password"] = getpass.getpass("   Password: ")
    return config


def load_telemetry(config: dict[str, str]) -> pd.DataFrame:
    """Carga la telemetría cruda de traffic_data vía SQLAlchemy."""
    conn_str = (
        f"postgresql://{config['user']}:{config['password']}"
        f"@{config['host']}:{config['port']}/{config['dbname']}"
    )
    engine = create_engine(conn_str)

    query = """
        SELECT id, clip_id, record_time, avg_speed,
               count_car, count_truck, count_bus,
               count_motorcycle, count_bicycle, total_vehicles
        FROM traffic_data
        ORDER BY record_time
    """
    df = pd.read_sql(text(query), engine)
    engine.dispose()
    return df


# Ejecución
try:
    db_config = get_db_config()
    df_raw = load_telemetry(db_config)
    # Guardar copia cruda para fallback futuro (solo las 10 columnas originales)
    df_raw.to_csv(RAW_CSV_PATH, index=False)
    print(f"✅ Telemetría cargada: {df_raw.shape[0]} registros, {df_raw.shape[1]} columnas")
    print(f"   Rango temporal: {df_raw['record_time'].min()} → {df_raw['record_time'].max()}")
    print(f"   CSV crudo guardado → {os.path.abspath(RAW_CSV_PATH)}")
    print(f"\n📊 Resumen estadístico:")
    display(df_raw.describe().round(2)) if "display" in dir() else print(df_raw.describe().round(2))
except Exception as e:
    print(f"🔴 Error al conectar a la BD: {e}")
    print("   Intentando cargar desde CSV crudo como fallback...")
    if os.path.exists(RAW_CSV_PATH):
        df_raw = pd.read_csv(RAW_CSV_PATH)
        print(f"✅ CSV crudo cargado: {df_raw.shape[0]} registros")
    else:
        raise RuntimeError(
            "No hay conexión a BD ni CSV crudo local. "
            "Restaura traffic_data.backup o configura las credenciales."
        )

## Ingeniería de Features — De 9 Campos Crudos a 14 Features

La telemetría cruda (velocidad + conteos) no captura relaciones entre registros consecutivos ni patrones temporales. La ingeniería de features expande los 9 campos originales a 14 variables que el modelo puede explotar:

| Feature | Origen | Justificación de dominio |
|---|---|---|
| `avg_speed` | Directo | Indicador primario de flujo vehicular |
| `total_vehicles` | Directo | Volumen absoluto de tráfico |
| `count_car` ... `count_bicycle` | Directo (5) | Composición vehicular — camiones y buses impactan diferente que autos |
| `heavy_vehicle_ratio` | Derivado | Proporción de vehículos pesados — tráfico pesado degrada más el flujo |
| `delta_speed` | Derivado (diff) | Aceleración/desaceleración entre minutos consecutivos |
| `delta_count` | Derivado (diff) | Tasa de cambio en volumen — detecta acumulación |
| `transition_flag` | Derivado | Señal binaria: cambios bruscos simultáneos en velocidad y volumen |
| `speed_variance` | Derivado (rolling) | Variabilidad reciente — tráfico inestable vs estable |
| `hour_of_day` | Temporal | Patrones circadianos de tráfico (rush hour, nocturno) |
| `weather_condition` | Simulado | Proxy de condición ambiental basado en hora (nocturno=riesgo) |

Los features derivados (`delta_*`, `speed_variance`) introducen NaN en los primeros registros que se eliminan.

In [ ]:
# Cell 3 — Ingeniería de Features

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Transforma 9 campos crudos en 14 features para el clasificador.

    Args:
        df: DataFrame con columnas de traffic_data.

    Returns:
        DataFrame con 14 features, sin NaN.
    """
    out = df.copy()

    # Features derivadas 
    # Ratio de vehículos pesados (trucks + buses / total)
    out["heavy_vehicle_ratio"] = (
        (out["count_truck"] + out["count_bus"])
        / out["total_vehicles"].clip(lower=1)
    )

    # Deltas inter-registro (aceleración / tasa de cambio)
    out["delta_speed"] = out["avg_speed"].diff()
    out["delta_count"] = out["total_vehicles"].diff()

    # Flag de transición: cambio brusco simultáneo en velocidad Y volumen
    out["transition_flag"] = (
        (out["delta_speed"].abs() > 10) & (out["delta_count"].abs() > 5)
    ).astype(int)

    # Variabilidad reciente de velocidad (ventana de 5 minutos)
    out["speed_variance"] = out["avg_speed"].rolling(
        window=5, min_periods=1
    ).std()

    # Hora del día (patrón circadiano)
    if pd.api.types.is_datetime64_any_dtype(out["record_time"]):
        out["hour_of_day"] = out["record_time"].dt.hour
    else:
        out["record_time"] = pd.to_datetime(out["record_time"])
        out["hour_of_day"] = out["record_time"].dt.hour

    # Condición meteorológica simulada (proxy por hora)
    # 0 = despejado (6-18h), 1 = nocturno/riesgo (resto)
    out["weather_condition"] = (
        ~out["hour_of_day"].between(6, 18)
    ).astype(int)

    # Eliminar filas con NaN de diff()
    out = out.dropna(subset=["delta_speed", "delta_count"]).reset_index(drop=True)

    return out


# Ejecución
df_features = engineer_features(df_raw)

# Guardar CSV de features para reproducibilidad (NO usar como fallback de datos crudos)
csv_path = os.path.join(DATA_DIR, "traffic_telemetry.csv")
df_features.to_csv(csv_path, index=False)

print(f"✅ Features engineered: {df_features.shape[0]} registros × {df_features.shape[1]} columnas")
print(f"   CSV features guardado → {os.path.abspath(csv_path)}")
print(f"\n📊 Correlación con avg_speed:")
FEATURE_COLS: list[str] = [
    "avg_speed", "total_vehicles",
    "count_car", "count_truck", "count_bus", "count_motorcycle", "count_bicycle",
    "heavy_vehicle_ratio", "delta_speed", "delta_count",
    "transition_flag", "speed_variance", "hour_of_day", "weather_condition",
]
corr = df_features[FEATURE_COLS].corr()["avg_speed"].drop("avg_speed").sort_values()
print(corr.to_string())

## Auto-Labeling — Reglas de Ingeniería de Tránsito

Sin anotación manual de miles de registros, usamos reglas de ingeniería como proxy de ground truth. Los umbrales se derivan de la normativa de operación del puente y la experiencia del dominio:

- **Accidente (3)** — el más severo: velocidad ~0 km/h tras una frenada brusca (`delta_speed < -20`) detectada dentro de una ventana reciente de 5 minutos, sostenida ≥3 registros consecutivos a <2 km/h. Asignado primero para que no sea sobreescrito.
- **Atascado (2)**: velocidad <5 km/h con alta densidad (>25 veh/min) sostenida ≥2 registros.
- **Reducido (1)**: velocidad entre 5-40 km/h con densidad moderada (15-25 veh/min). Como cada registro es 1 minuto, la persistencia >60s ya está implícita.
- **Normal (0)**: todo lo demás (default catch-all).

**Limitación conocida**: estas etiquetas NO son ground truth humano. Un operador SISE corrige esto en la Fase 2 futura via HITL (campos `is_human_validated` y `human_override_state` en la tabla `traffic_classifications`). Ver [BIAS_AND_LIMITATIONS.md](../../docs/BIAS_AND_LIMITATIONS.md) §6.

In [ ]:
# Cell 4 — Auto-Labeling + Distribución de Clases

STATE_LABELS: dict[int, str] = {
    0: "Normal",
    1: "Reducido",
    2: "Atascado",
    3: "Accidente",
}


def assign_traffic_state(df: pd.DataFrame) -> pd.Series:
    """Asigna estados de tráfico usando reglas de ingeniería.

    Orden de evaluación: severo primero (Accidente → Atascado → Reducido).
    Normal es el default (código 0).

    La detección de Accidente usa un modelo de dos fases:
      1. Impacto: frenada brusca (delta_speed < -20) dentro de una ventana
         reciente de 5 registros.
      2. Persistencia: velocidad < 2 km/h sostenida ≥ 3 registros consecutivos.

    Args:
        df: DataFrame con features engineered.

    Returns:
        Series con códigos de estado (0-3).
    """
    states = pd.Series(0, index=df.index, dtype=int)  # Default: Normal

    # Accidente (3): impacto reciente + velocidad ~0 sostenida 
    low_speed = df["avg_speed"] < 2
    # Fase 1 — ¿Hubo frenada brusca en los últimos 5 minutos?
    braking = df["delta_speed"] < -20
    had_recent_braking = braking.rolling(window=5, min_periods=1).max().astype(bool)
    # Fase 2 — ¿Lleva ≥3 registros consecutivos a velocidad ~0?
    consecutive_low = low_speed.rolling(window=3, min_periods=3).sum() >= 3
    accident_mask = low_speed & had_recent_braking & consecutive_low
    states[accident_mask] = 3

    # Atascado (2): congestión sostenida 
    congestion = (df["avg_speed"] < 5) & (df["total_vehicles"] > 25)
    consecutive_congestion = congestion.rolling(window=2, min_periods=2).sum() >= 2
    stuck_mask = congestion & consecutive_congestion & (states != 3)
    states[stuck_mask] = 2

    # Reducido (1): flujo degradado 
    reduced_mask = (
        df["avg_speed"].between(5, 40)
        & df["total_vehicles"].between(15, 25)
        & (states == 0)  # Solo si no fue Accidente ni Atascado
    )
    states[reduced_mask] = 1

    return states


# Ejecución 
df_features["traffic_state"] = assign_traffic_state(df_features)

# Distribución 
dist = df_features["traffic_state"].value_counts().sort_index()
print("📊 Distribución de estados de tráfico:")
for code, count in dist.items():
    pct = 100 * count / len(df_features)
    print(f"   {STATE_LABELS[code]:>10} ({code}): {count:>5} registros ({pct:.1f}%)")

# Verificar que haya al menos 2 clases
n_classes = dist.index.nunique()
if n_classes < 2:
    print("🔴 Solo se encontró 1 clase. Los umbrales no discriminan en este dataset.")
else:
    print(f"\n✅ {n_classes} clases detectadas")

# Clases sin muestras
for code, label in STATE_LABELS.items():
    if code not in dist.index:
        print(f"⚠️  Clase '{label}' ({code}) sin muestras — se excluirá del entrenamiento")

# Visualización 
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#2ecc71", "#f39c12", "#e74c3c", "#8e44ad"]
bars = ax.bar(
    [STATE_LABELS[c] for c in sorted(dist.index)],
    [dist[c] for c in sorted(dist.index)],
    color=[colors[c] for c in sorted(dist.index)],
)
ax.set_ylabel("Registros")
ax.set_title("Distribución de Estados de Tráfico (Auto-Labeling)")
for bar, count in zip(bars, [dist[c] for c in sorted(dist.index)]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(count), ha="center", fontsize=10)
plt.tight_layout()
plt.show()

## Balanceo y Partición — SMOTE + Estratificación

El dataset real está fuertemente desbalanceado: se espera ~80% Normal, con Accidente potencialmente <1%. Entrenar un modelo directamente produciría un clasificador que ignora las clases minoritarias.

**Estrategia**:
1. **StandardScaler**: Normaliza features a media=0, std=1 (requerido para redes neuronales)
2. **Train/Test split** (80/20): Estratificado para mantener proporciones originales en ambos sets
3. **SMOTE** (Synthetic Minority Over-sampling Technique): Se aplica **solo al training set** para generar muestras sintéticas de clases minoritarias. El test set permanece intacto como evaluación realista

El scaler se exporta como artefacto (`feature_scaler.joblib`) para que la inferencia en producción use la misma transformación.

In [ ]:
# Cell 5 — SMOTE + Train/Test Split

# Preparar matrices 
X = df_features[FEATURE_COLS].values
y = df_features["traffic_state"].values

# Escalado 
scaler = StandardScaler()

# Train/Test split estratificado 
# Guard: stratify falla si alguna clase tiene <2 muestras
class_counts = np.bincount(y)
min_samples_per_class = class_counts[class_counts > 0].min()
can_stratify = min_samples_per_class >= 2

if can_stratify:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
    )
else:
    print("⚠️ Clase con <2 muestras — split sin estratificación")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_SEED
    )

# Fit scaler SOLO en training, transformar ambos
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"📊 Partición original:")
print(f"   Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"   Train distribución: {dict(zip(*np.unique(y_train, return_counts=True)))}")

# SMOTE (solo en training) 
# Verificar que la clase minoritaria tenga al menos k_neighbors+1 muestras
train_counts = np.bincount(y_train)
min_class_count = train_counts[train_counts > 0].min()
k_neighbors = min(5, min_class_count - 1) if min_class_count > 1 else 1

if min_class_count < 2:
    print("⚠️ Clase con <2 muestras en train. SMOTE desactivado — se procede sin balanceo.")
    X_train_res, y_train_res = X_train, y_train
else:
    sm = SMOTE(random_state=RANDOM_SEED, k_neighbors=k_neighbors)
    X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
    print(f"\n✅ SMOTE aplicado (k_neighbors={k_neighbors}):")
    print(f"   Train balanceado: {X_train_res.shape[0]} muestras")
    print(f"   Distribución: {dict(zip(*np.unique(y_train_res, return_counts=True)))}")

# Exportar scaler 
scaler_path = os.path.join(MODEL_DIR, "feature_scaler.joblib")
joblib.dump(scaler, scaler_path)
print(f"\n💾 Scaler guardado → {os.path.abspath(scaler_path)}")

## Arquitectura del Modelo — MLP Tabular (Fase 1)

El modelo es un **Perceptrón Multicapa (MLP)** implementado con `tf.keras.Sequential`. La arquitectura es deliberadamente simple — esta es la Fase 1 diseñada para validar el pipeline completo. La Fase 2 evoluciona a LSTM con memoria temporal.

**¿Por qué estas dimensiones?**
- **Dense(64)**: Capa de entrada con capacidad suficiente para aprender combinaciones no lineales de 14 features
- **Dense(32)**: Capa de compresión que fuerza representaciones más abstractas
- **BatchNormalization**: Estabiliza y acelera el entrenamiento normalizando activaciones entre capas
- **Dropout(0.3 → 0.2)**: Regularización decreciente — más agresiva cerca del input (donde hay más redundancia)
- **Softmax(n_classes)**: Distribución de probabilidad sobre los 4 estados

In [ ]:
# Cell 6 — Definición del Modelo + Entrenamiento

# Número de clases dinámico (puede ser <4 si algún estado no tiene muestras)
n_classes: int = len(np.unique(y))
n_features: int = X_train_res.shape[1]

print(f"🏗️ Construyendo modelo MLP: {n_features} features → {n_classes} clases")

# Arquitectura 
model = Sequential([
    Input(shape=(n_features,)),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation="relu"),
    BatchNormalization(),
    Dropout(0.2),
    Dense(n_classes, activation="softmax"),
], name="traffic_state_classifier")

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

# Callbacks 
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        patience=5,
        factor=0.5,
        min_lr=1e-6,
        verbose=1,
    ),
]

# Entrenamiento 
print("\n🚀 Iniciando entrenamiento...")
history = model.fit(
    X_train_res,
    y_train_res,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1,
)

# Visualización del entrenamiento 
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(history.history["loss"], label="Train Loss")
ax1.plot(history.history["val_loss"], label="Val Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Pérdida durante entrenamiento")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(history.history["accuracy"], label="Train Accuracy")
ax2.plot(history.history["val_accuracy"], label="Val Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Precisión durante entrenamiento")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_epoch = np.argmin(history.history["val_loss"]) + 1
print(f"\n✅ Entrenamiento completado — mejor epoch: {best_epoch}")

## Evaluación — Métricas de Clasificación

Las métricas clave para este clasificador son:

- **F1-macro** ≥ 0.85: Promedio no ponderado del F1 por clase. Penaliza equitativamente el rendimiento en clases raras (Accidente) y frecuentes (Normal)
- **Recall por clase** > 0: Especialmente para Accidente — un recall de 0 significaría que el modelo nunca detecta esta clase crítica
- **Confusion matrix**: Identifica confusiones sistemáticas (ej: Normal↔Reducido es el par más probable de confusión)

El modelo se exporta como `.keras` (formato nativo autónomo) junto con el mapping de labels.

In [ ]:
# Cell 7 — Evaluación + Exportación del Modelo

# Predicción en test set 
y_proba = model.predict(X_test)
y_pred = y_proba.argmax(axis=1)

# Nombres de clases presentes 
present_classes = sorted(np.unique(np.concatenate([y_test, y_pred])))
target_names = [STATE_LABELS[c] for c in present_classes]

# Classification Report 
print("=" * 60)
print("REPORTE DE CLASIFICACIÓN")
print("=" * 60)
report = classification_report(
    y_test, y_pred,
    labels=present_classes,
    target_names=target_names,
    zero_division=0,
)
print(report)

# F1-macro
f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
print(f"{'F1-macro':>15}: {f1_macro:.4f}")

if f1_macro >= 0.85:
    print(f"✅ F1-macro CUMPLE el target (≥ 0.85)")
else:
    print(f"⚠️  F1-macro POR DEBAJO del target (≥ 0.85) — revisar balanceo o umbrales")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=present_classes)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=target_names,
    yticklabels=target_names,
    ax=ax,
)
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
ax.set_title(f"Confusion Matrix — F1-macro: {f1_macro:.4f}")
plt.tight_layout()
plt.show()

# Recall por clase
print("\n📊 Recall por clase:")
for i, cls in enumerate(present_classes):
    row_sum = cm[i].sum()
    recall = cm[i, i] / row_sum if row_sum > 0 else 0.0
    status = "✅" if recall > 0 else "🔴"
    print(f"   {status} {STATE_LABELS[cls]:>10}: {recall:.4f}")

# Exportar modelo
model_path = os.path.join(MODEL_DIR, "traffic_classifier.keras")
model.save(model_path)

# Exportar label_mapping filtrado a las clases que el modelo realmente vio
label_mapping = {c: STATE_LABELS[c] for c in present_classes}
label_path = os.path.join(MODEL_DIR, "label_mapping.joblib")
joblib.dump(label_mapping, label_path)

print(f"\n💾 Artefactos exportados:")
print(f"   Modelo  → {os.path.abspath(model_path)} ({os.path.getsize(model_path) / 1024:.1f} KB)")
print(f"   Labels  → {os.path.abspath(label_path)} (clases: {list(label_mapping.values())})")
print(f"   Scaler  → {os.path.abspath(os.path.join(MODEL_DIR, 'feature_scaler.joblib'))}")

## Persistencia — Dos Tablas Nuevas con FK

El resultado del clasificador se persiste en PostgreSQL con trazabilidad completa:

```
traffic_data (legacy, intocable)
    ↓ FK: source_record_id
telemetry_raw (14 features engineered)
    ↓ FK: telemetry_id
traffic_classifications (predicción + HITL)
```

- **`telemetry_raw`**: Almacena los 14 features calculados, con FK al registro original en `traffic_data`. Permite reproducir el entrenamiento y auditar qué datos alimentaron cada predicción.
- **`traffic_classifications`**: Almacena la predicción del modelo, la confianza, la versión del modelo, y campos HITL (`is_human_validated`, `human_override_state`, `validated_at`) que se activarán cuando un operador SISE pueda confirmar/descartar alertas.

La persistencia es **opcional**: si no hay BD configurada, el notebook funciona completo y exporta los artefactos localmente.

In [ ]:
# Cell 8 — Crear Tablas + Persistir Resultados

DDL_TELEMETRY_RAW: str = """
CREATE TABLE IF NOT EXISTS telemetry_raw (
    id SERIAL PRIMARY KEY,
    source_record_id INTEGER REFERENCES traffic_data(id),
    record_time TIMESTAMP NOT NULL,
    avg_speed NUMERIC(5,2),
    total_vehicles INTEGER,
    count_car INTEGER,
    count_truck INTEGER,
    count_bus INTEGER,
    count_motorcycle INTEGER,
    count_bicycle INTEGER,
    heavy_vehicle_ratio NUMERIC(5,4),
    delta_speed NUMERIC(6,2),
    delta_count INTEGER,
    transition_flag SMALLINT DEFAULT 0,
    speed_variance NUMERIC(6,2),
    hour_of_day SMALLINT,
    weather_condition SMALLINT DEFAULT 0,
    UNIQUE (source_record_id)
);
"""

DDL_TRAFFIC_CLASSIFICATIONS: str = """
CREATE TABLE IF NOT EXISTS traffic_classifications (
    id SERIAL PRIMARY KEY,
    telemetry_id INTEGER REFERENCES telemetry_raw(id),
    classified_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    traffic_state SMALLINT NOT NULL,
    state_label TEXT NOT NULL,
    confidence NUMERIC(5,4) NOT NULL,
    model_version TEXT NOT NULL,
    is_human_validated BOOLEAN DEFAULT FALSE,
    human_override_state SMALLINT,
    validated_at TIMESTAMP,
    UNIQUE (telemetry_id, model_version)
);
"""

MODEL_VERSION: str = "mlp-v1.0"

TELEMETRY_COLS: list[str] = [
    "source_record_id", "record_time", "avg_speed", "total_vehicles",
    "count_car", "count_truck", "count_bus", "count_motorcycle",
    "count_bicycle", "heavy_vehicle_ratio", "delta_speed", "delta_count",
    "transition_flag", "speed_variance", "hour_of_day", "weather_condition",
]

INSERT_TELEMETRY_SQL: str = """
    INSERT INTO telemetry_raw (
        source_record_id, record_time, avg_speed, total_vehicles,
        count_car, count_truck, count_bus, count_motorcycle,
        count_bicycle, heavy_vehicle_ratio, delta_speed, delta_count,
        transition_flag, speed_variance, hour_of_day, weather_condition
    ) VALUES (
        :source_record_id, :record_time, :avg_speed, :total_vehicles,
        :count_car, :count_truck, :count_bus, :count_motorcycle,
        :count_bicycle, :heavy_vehicle_ratio, :delta_speed, :delta_count,
        :transition_flag, :speed_variance, :hour_of_day, :weather_condition
    )
    ON CONFLICT (source_record_id) DO NOTHING
"""

INSERT_CLASSIFICATION_SQL: str = """
    INSERT INTO traffic_classifications (
        telemetry_id, traffic_state, state_label,
        confidence, model_version
    ) VALUES (
        :telemetry_id, :traffic_state, :state_label,
        :confidence, :model_version
    )
    ON CONFLICT (telemetry_id, model_version) DO NOTHING
"""


def persist_results(
    df: pd.DataFrame,
    model: tf.keras.Model,
    scaler: StandardScaler,
    config: dict[str, str],
    feature_cols: list[str],
) -> None:
    """Crea tablas y persiste features + clasificaciones en PostgreSQL.

    Usa inserciones por lote (executemany via conn.execute con lista de dicts)
    en lugar de row-by-row para mejor rendimiento.

    Args:
        df: DataFrame con features engineered y traffic_state.
        model: Modelo Keras entrenado.
        scaler: StandardScaler fitted.
        config: Credenciales de BD.
        feature_cols: Lista de nombres de features.
    """
    conn_str = (
        f"postgresql://{config['user']}:{config['password']}"
        f"@{config['host']}:{config['port']}/{config['dbname']}"
    )
    engine = create_engine(conn_str)

    with engine.begin() as conn:
        # Crear tablas
        conn.execute(text(DDL_TELEMETRY_RAW))
        conn.execute(text(DDL_TRAFFIC_CLASSIFICATIONS))
        print("✅ Tablas creadas (o ya existían)")

        # Preparar datos de telemetry_raw 
        df_telemetry = df.rename(columns={"id": "source_record_id"})[
            [c for c in TELEMETRY_COLS if c in df.columns or c == "source_record_id"]
        ].copy()

        # Asegurar tipos correctos para PostgreSQL
        df_telemetry["delta_count"] = df_telemetry["delta_count"].astype(int)
        df_telemetry["transition_flag"] = df_telemetry["transition_flag"].astype(int)
        df_telemetry["hour_of_day"] = df_telemetry["hour_of_day"].astype(int)
        df_telemetry["weather_condition"] = df_telemetry["weather_condition"].astype(int)

        # Insertar telemetry_raw por lote
        telemetry_records = df_telemetry.to_dict(orient="records")
        result = conn.execute(text(INSERT_TELEMETRY_SQL), telemetry_records)
        print(f"📊 telemetry_raw: {len(telemetry_records)} registros enviados")

        # Clasificar todo el dataset
        X_all = scaler.transform(df[feature_cols].values)
        proba_all = model.predict(X_all, verbose=0)
        pred_all = proba_all.argmax(axis=1)
        conf_all = proba_all.max(axis=1)

        # Obtener IDs de telemetry_raw
        telemetry_ids = conn.execute(
            text("SELECT id, source_record_id FROM telemetry_raw ORDER BY id")
        ).fetchall()
        source_to_telemetry = {row[1]: row[0] for row in telemetry_ids}

        # Preparar datos de classifications por lote
        classification_records: list[dict] = []
        for idx, (_, row) in enumerate(df.iterrows()):
            source_id = row.get("id")
            telemetry_id = source_to_telemetry.get(source_id)
            if telemetry_id is None:
                continue
            classification_records.append({
                "telemetry_id": int(telemetry_id),
                "traffic_state": int(pred_all[idx]),
                "state_label": STATE_LABELS[int(pred_all[idx])],
                "confidence": float(round(conf_all[idx], 4)),
                "model_version": MODEL_VERSION,
            })

        if classification_records:
            conn.execute(text(INSERT_CLASSIFICATION_SQL), classification_records)

        print(f"📊 traffic_classifications: {len(classification_records)} registros enviados")

    engine.dispose()

    # Resumen
    print(f"\n✅ Persistencia completada (model_version={MODEL_VERSION})")
    dist = pd.Series(pred_all).value_counts().sort_index()
    for code, count in dist.items():
        print(f"   {STATE_LABELS[code]:>10}: {count} clasificaciones")


# Ejecución 
try:
    persist_results(df_features, model, scaler, db_config, FEATURE_COLS)
except NameError:
    print("⚠️ Sin configuración de BD — resultados solo locales")
    print("   Los artefactos del modelo están disponibles en:")
    print(f"   {os.path.abspath(MODEL_DIR)}")
except Exception as e:
    print(f"🔴 Error de persistencia: {e}")
    print("   El modelo y artefactos están disponibles localmente")